### Dataset Download Instructions

This project uses the EEG Motor Imagery dataset from:

**Kaya et al. (2018)**  
https://doi.org/10.1038/sdata.2018.211

### Steps to Run:

1. Download the `.mat` files CLASubjectE from the dataset repository. Note: This can be extended to other subjects as needed
2. Place the downloaded files in a local folder (e.g., `Downloads/`).
3. Update the file paths in the code below:
4. Run the code

```python
mat_file_paths = [
    r"path_to_your_folder/CLASubjectE1601223StLRHand.mat",
    r"path_to_your_folder/CLASubjectE1601193StLRHand.mat",
    r"path_to_your_folder/CLASubjectE1512253StLRHand.mat",
]

In [ ]:
import pandas as pd
import numpy as np
import scipy.io
import os

# Define channel names and remove unwanted channels
channel_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
                 'A1', 'A2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'X5']
channels_to_remove = ['A1', 'A2', 'X5']

# Indices of channels to keep
channels_to_keep = [i for i, ch in enumerate(channel_names) if ch not in channels_to_remove]

def process_data(mat_data):
    o_data = mat_data['o'][0, 0]

    if 'data' in o_data.dtype.names:
        data_key = o_data['data'][:, channels_to_keep]  # Keep only relevant channels
        marker_data = o_data['marker']

        print(f"Shape of 'data' after channel filtering: {data_key.shape}")

        df_labels = pd.DataFrame(marker_data)
        labels_array = df_labels.to_numpy()

        class_info = []
        for sampli in range(0, labels_array.shape[0] - 1):
            if labels_array[sampli] == 0 and labels_array[sampli + 1] == 1:
                class_info.append([1, sampli + 1])
            elif labels_array[sampli] == 0 and labels_array[sampli + 1] == 2:
                class_info.append([2, sampli + 1])

        class_info_array = np.array(class_info)

        return class_info_array, data_key, marker_data

    else:
        print("'data' not found in 'o_data'.")
        return None, None, None

mat_file_paths = [
    r"C:\Users\HP Pavilion\Downloads\CLASubjectE1601223StLRHand.mat",
    r"C:\Users\HP Pavilion\Downloads\CLASubjectE1601193StLRHand.mat",
    r"C:\Users\HP Pavilion\Downloads\CLASubjectE1512253StLRHand.mat",
]

output_folder = r'D:\paper eeg'
os.makedirs(output_folder, exist_ok=True)

for i, mat_file_path in enumerate(mat_file_paths):
    mat_data = scipy.io.loadmat(mat_file_path)
    class_info_array, data_key, marker_data = process_data(mat_data)

    if data_key is not None and class_info_array is not None:
        print(f"\nProcessing dataset {i + 1}...")

        rows_list = []

        for class_info in class_info_array:
            class_label, onset_sample = class_info
            start_index = max(0, onset_sample - 200)
            end_index = min(len(data_key) - 1, onset_sample + 200)

            start_value = data_key[start_index:onset_sample, :].flatten()
            end_value = data_key[onset_sample + 1:end_index + 1, :].flatten()

            start_value_reshaped = start_value.reshape(1, -1)
            end_value_reshaped = end_value.reshape(1, -1)

            label_window_startvalues = marker_data[onset_sample - 1, :1]
            label_window_endvalues = marker_data[onset_sample + 1, :1]

            row_start = np.concatenate([start_value_reshaped.flatten(), label_window_startvalues.flatten()])
            row_end = np.concatenate([end_value_reshaped.flatten(), label_window_endvalues.flatten()])
            rows_list.extend([row_start, row_end])

        df = pd.DataFrame(rows_list)

        output_file_path = os.path.join(output_folder, f'dataset_{i + 1}_eeg_features.csv')
        df.to_csv(output_file_path, index=False, header=False)

        print(f"Saved EEG features to: {output_file_path}")

    else:
        print(f"Skipping dataset {i + 1} due to missing data or labels.")


Shape of 'data' after channel filtering: (667000, 19)

Processing dataset 1...
Saved EEG features to: D:\paper eeg\dataset_1_eeg_features.csv
Shape of 'data' after channel filtering: (664400, 19)

Processing dataset 2...
Saved EEG features to: D:\paper eeg\dataset_2_eeg_features.csv
Shape of 'data' after channel filtering: (664000, 19)

Processing dataset 3...
Saved EEG features to: D:\paper eeg\dataset_3_eeg_features.csv




*   We define 22 EEG channel names.

*   We remove unwanted channels (A1, A2, X5).

*  The remaining channels are used for feature extraction



*   process_data(mat_data) Function

This function:

Extracts:
EEG signal data (o_data['data'])
Marker labels (o_data['marker'])




Filters:
Keeps only selected EEG channels.
Finds event onsets where:
Marker changes from 0 → 1 → Class 1
Marker changes from 0 → 2 → Class 2

It returns:

class_info_array → [class_label, onset_sample]
data_key → EEG data
marker_data → labels

We then extract:

200 samples before event
200 samples after event

For each window:

EEG signals are flattened into a 1D vector.
Corresponding marker label is appended at the end.

So each event produces:




*   One row for before onset
*   One row for after onset




Features = 200 samples × number_of_channels

lastly, save combined dataset in the desired folder in csv format


In [ ]:
import pandas as pd
import numpy as np
import scipy.io
import os

# Define EEG channel names
channel_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
                 'A1', 'A2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'X5']
channels_to_remove = ['A1', 'A2', 'X5']

# Get indices of channels to keep
channels_to_keep = [i for i, ch in enumerate(channel_names) if ch not in channels_to_remove]

def process_data(mat_data):
    """Extracts EEG data and markers from .mat files."""
    o_data = mat_data['o'][0, 0]

    if 'data' in o_data.dtype.names:
        data_key = o_data['data'][:, channels_to_keep]  # Keep only selected channels
        marker_data = o_data['marker']

        df_labels = pd.DataFrame(marker_data)
        labels_array = df_labels.to_numpy()

        class_info = []
        for sampli in range(0, labels_array.shape[0] - 1):
            if labels_array[sampli] == 0 and labels_array[sampli + 1] == 1:
                class_info.append([1, sampli + 1])
            elif labels_array[sampli] == 0 and labels_array[sampli + 1] == 2:
                class_info.append([2, sampli + 1])

        class_info_array = np.array(class_info)

        return class_info_array, data_key, marker_data
    else:
        print("'data' not found in 'o_data'.")
        return None, None, None

mat_file_paths = [
    r"C:\Users\HP Pavilion\Downloads\CLASubjectE1601223StLRHand.mat",
    r"C:\Users\HP Pavilion\Downloads\CLASubjectE1601193StLRHand.mat",
    r"C:\Users\HP Pavilion\Downloads\CLASubjectE1512253StLRHand.mat",
]

output_folder = r'D:\paper eeg'
os.makedirs(output_folder, exist_ok=True)

# Combined dataset storage
combined_rows = []

for i, mat_file_path in enumerate(mat_file_paths):
    mat_data = scipy.io.loadmat(mat_file_path)
    class_info_array, data_key, marker_data = process_data(mat_data)

    if data_key is not None and class_info_array is not None:
        print(f"\nProcessing dataset {i + 1}...")

        for class_info in class_info_array:
            class_label, onset_sample = class_info
            start_index = max(0, onset_sample - 200)
            end_index = min(len(data_key) - 1, onset_sample + 200)

            start_value = data_key[start_index:onset_sample, :].flatten()
            end_value = data_key[onset_sample + 1:end_index + 1, :].flatten()

            label_window_startvalues = marker_data[onset_sample - 1, :1]
            label_window_endvalues = marker_data[onset_sample + 1, :1]

            row_start = np.concatenate([start_value, label_window_startvalues.flatten()])
            row_end = np.concatenate([end_value, label_window_endvalues.flatten()])
            combined_rows.extend([row_start, row_end])

    else:
        print(f"Skipping dataset {i + 1} due to missing data or labels.")

# Convert combined dataset to DataFrame
if combined_rows:
    df_combined = pd.DataFrame(combined_rows)

    # Save the final combined dataset
    combined_output_file = os.path.join(output_folder, 'combined_eeg_features.csv')
    df_combined.to_csv(combined_output_file, index=False, header=False)

    # Print final shape of dataset
    print(f"\n Combined EEG features saved to: {combined_output_file}")
    print(f" Final dataset shape: {df_combined.shape}")  # Print dataset dimensions
else:
    print("\n No valid data found. Combined dataset not created.")



Processing dataset 1...

Processing dataset 2...

Processing dataset 3...

✅ Combined EEG features saved to: D:\paper eeg\combined_eeg_features.csv
📏 Final dataset shape: (3808, 3801)
